# 21 Sep

## wonko script to give entry accepted name and assign rest as synonyms

- am still keeping the origional scientific name so that I can match compounds and plants later

In [ ]:
import pandas as pd

merged_plants_file = ''
wcvp_file = '' 
error_file = ''
output_file = ''

df_merged = pd.read_csv(merged_plants_file)
df_wcvp = pd.read_csv(wcvp_file, sep='|')

df_merged['Unnamed: 0'] = df_merged['temp_id']
df_merged['acceptednameusageid'] = pd.to_numeric(df_merged['acceptednameusageid'],errors='coerce').astype('Int64')
df_wcvp['taxonid'] = pd.to_numeric(df_wcvp['taxonid'], errors='coerce').astype('Int64')


def get_properName(accepted_id):
    accepted_id = int(accepted_id)
    row = df_wcvp.loc[df_wcvp['taxonid'] == accepted_id]
    if row.empty:
        return None
    else:
        return row.iloc[0]

print('Begining official name assignment', flush=True)

entry = []
errors = []
count = 0
total = len(df_merged)
for idx,row in df_merged.iterrows():
    count += 1
    proper_id = row.acceptednameusageid

    if pd.notna(proper_id):
        proper_data = get_properName(proper_id)
        if proper_data is None:
            errors.append([row.original_scientificName, row.ID, 'ID Not found in wcvp apparently'])
            continue
        plant_id = row.temp_id
        accepted_id = proper_id
        name = proper_data['scientfiicname']
        family = proper_data['family']
        authorship = proper_data['scientfiicnameauthorship']
        namepublishedin = proper_data['namepublishedin']
        powo_id_accepted = proper_data['scientificnameid']
        origional_coconut_name = row.original_scientificName
        gbif_name = row.scientificName
        gbif_ID = row.ID 
        gbif_accepted_name = row.acceptedScientificName
        gbif_acceptef_id = row.acceptedID
        powo_scientific_name = row.scientfiicname
        powo_id = row.taxonid
        entry.append([plant_id, accepted_id, name, family, authorship, namepublishedin, powo_id_accepted, origional_coconut_name, gbif_name, gbif_ID, gbif_accepted_name, gbif_acceptef_id, powo_scientific_name, powo_id])
    else:
        errors.append([row.original_scientificName, row.ID, 'No acceepted usage name ID'])

    prog = count/total*100

    if count % 1000 == 0:
        print(f'Progress: {prog:.2f}%', flush=True)

cols = ['plant_id','accepted_id_wcvp', 'name', 'family', 'authorship', 'namepublishedin', 'powo_id_accepted', 'original_coconut_name', 'gbif_name', 'gbif_ID', 'gbif_accepted_name', 'gbif_accepted_id', 'powo_scientific_name', 'powo_id'] 
df_dups = pd.DataFrame(entry, columns= cols)

df_errors = pd.DataFrame(errors,columns=['original_scientificName', 'gbif_ID', 'error'])
df_errors.to_csv(error_file, index=False)

def join_dedup(things):
    out = []
    added = set()
    for thing in things:
        if pd.isna(thing):
            continue
        thing = str(thing).strip()
        if not thing or thing not in added:
            added.add(thing)
            out.append(thing)
    return '|'.join(out)  
        


out_df = (df_dups.groupby('accepted_id',sort=False,as_index=False).agg(join_dedup))
out_df.drop(columns=['plant_id'],inplace=True)
out_df['plant_id'] = [num for num in range(1,len(out_df)+1)]
out_df.insert(0,'plant_id',out_df.pop('plant_id'))
out_df.to_csv(output_file, index=False)

print(f'Input rows: {len(df_merged):,}')
print(f'Accepted-name rows retained: {len(df_dups):,}')
print(f'Accepted taxa output: {len(out_df):,}')
print(f'Errors: {len(df_errors):,}')

